In [1]:
import os, time, json, getpass, requests
import pandas as pd

URL = "https://merpis.ptmerahputih.com/api/porttoport"
TOKEN = (os.environ.get("MERPIS_TOKEN_PORTTOPORT") or os.environ.get("MERPIS_TOKEN_PROJECTS")
         or getpass.getpass("Token Bearer: ").strip())
headers = {"Accept": "application/json", "Authorization": f"Bearer {TOKEN}"}

t0 = time.time()
r = requests.get(URL, headers=headers, timeout=(10, 600))
print("Status:", r.status_code, "|", f"{time.time()-t0:.1f}s", "|", f"{len(r.content)/1e6:.3f} MB")
r.raise_for_status()
data = r.json()

with open("porttoport_raw.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False)

print("Tipe:", type(data).__name__)
if isinstance(data, dict):
    print({k: v for k, v in data.items() if not isinstance(v, (list, dict))})
print({k: v for k, v in r.headers.items() if k.lower() in ("link", "x-total-count", "x-total", "x-per-page")})

# Cari list of dict di level mana pun
def cari_list(obj, path="root"):
    if isinstance(obj, list) and obj and isinstance(obj[0], dict):
        yield path, obj
    elif isinstance(obj, dict):
        for k, v in obj.items():
            yield from cari_list(v, f"{path}.{k}")

kandidat = list(cari_list(data))
for path, lst in kandidat:
    print(f"{path:30s} {len(lst):7d} item | key: {list(lst[0].keys())[:10]}")

Token Bearer:  ········


Status: 200 | 0.7s | 0.036 MB
Tipe: dict
{'code': 200, 'message': 'success'}
{}
root.data.porttoport               138 item | key: ['id', 'pointa', 'pointb', 'distance', 'bbm', 'created_at', 'created_by', 'updated_at', 'active', 'approval_status']


In [2]:
import numpy as np

# Pakai list terbesar, ratakan, lalu pisahkan kolom nested (dilaporkan, tidak dibuang diam-diam)
path, records = max(kandidat, key=lambda x: len(x[1]))
print("Sumber list:", path, "|", len(records), "item")
df_raw = pd.json_normalize(records)

nested = [c for c in df_raw.columns if df_raw[c].apply(lambda x: isinstance(x, (list, dict))).any()]
for c in nested:
    mb = df_raw[c].apply(lambda x: len(json.dumps(x, default=str)) if isinstance(x, (list, dict)) else 0).sum() / 1e6
    print(f"Nested: {c:40s} {mb:.2f} MB")
d = df_raw.drop(columns=nested).copy()

# Cleansing
d.columns = [c.strip().lower().replace(".", "_").replace(" ", "_") for c in d.columns]
for c in d.select_dtypes(include=["object", "string"]).columns:
    d[c] = d[c].apply(lambda x: x.strip() if isinstance(x, str) else x)
    d[c] = d[c].replace({"": np.nan, "None": np.nan, "none": np.nan, "null": np.nan, "NaN": np.nan})

def ke_wib(s):
    """Nilai berakhiran Z (UTC) -> WIB. Tanpa zona waktu = dianggap sudah WIB, tidak diubah."""
    s = s.astype("object")
    is_z = s.notna() & s.astype(str).str.endswith("Z")
    z = pd.to_datetime(s.where(is_z), errors="coerce", utc=True).dt.tz_convert("Asia/Jakarta").dt.tz_localize(None)
    n = pd.to_datetime(s.where(~is_z), errors="coerce")
    return z.fillna(n)

for c in d.columns:
    if d[c].dtype != "object":
        continue
    if c.endswith("_at"):
        s = ke_wib(d[c])
        if d[c].notna().any() and s.notna().sum() >= d[c].notna().sum() * 0.9:
            d[c] = s
            continue
    if "date" in c or c.endswith("builtyear"):
        s = pd.to_datetime(d[c], errors="coerce")
        if d[c].notna().any() and s.notna().sum() >= d[c].notna().sum() * 0.9:
            d[c] = s
            continue

for c in d.columns:   # numerik hanya bila semua nilai terisi terbaca dan tidak ada angka berawalan 0
    if d[c].dtype != "object" or not d[c].notna().any():
        continue
    if d[c].dropna().astype(str).str.match(r"^0\d+").any():
        continue
    conv = pd.to_numeric(d[c], errors="coerce")
    if conv.notna().sum() == d[c].notna().sum():
        d[c] = conv

if "id" in d.columns:
    print("Duplikat id:", d["id"].duplicated().sum())
    d = d.drop_duplicates(subset="id", keep="first").sort_values("id", ascending=False)
d = d.reset_index(drop=True)
print("Shape:", df_raw.shape, "->", d.shape)

summary = pd.DataFrame({
    "kolom": d.columns,
    "tipe": d.dtypes.astype(str).values,
    "terisi": d.notna().sum().values,
    "kosong_%": (d.isna().mean() * 100).round(1).values,
    "unik": d.nunique().values,
    "contoh": [d[c].dropna().iloc[0] if d[c].notna().any() else None for c in d.columns],
})
pd.set_option("display.max_rows", None, "display.max_colwidth", 50, "display.width", 200)
display(summary)
print("Kosong semua:", d.columns[d.isna().all()].tolist())
print("Konstan     :", [c for c in d.columns if d[c].nunique(dropna=False) == 1 and c != "id"])

with pd.ExcelWriter("porttoport_cleansing_check.xlsx", engine="openpyxl") as w:
    d.to_excel(w, sheet_name="data_clean", index=False)
    summary.to_excel(w, sheet_name="ringkasan_kolom", index=False)
    for ws in w.sheets.values():
        ws.freeze_panes = "B2"
        ws.auto_filter.ref = ws.dimensions
print("Tersimpan: porttoport_cleansing_check.xlsx |", d.shape[0], "baris x", d.shape[1], "kolom")

Sumber list: root.data.porttoport | 138 item
Duplikat id: 0
Shape: (138, 12) -> (138, 12)


,kolom,tipe,terisi,kosong_%,unik,contoh
0,id,int64,138,0.0,138,138
1,pointa,int64,138,0.0,77,159
2,pointb,int64,138,0.0,29,70
3,distance,float64,138,0.0,78,535.0
4,bbm,object,0,100.0,0,None
5,created_at,datetime64[ns],138,0.0,1,2026-10-29 16:34:21
6,created_by,object,138,0.0,1,mfakhrurrozi
7,updated_at,datetime64[ns],138,0.0,2,2026-10-30 10:04:43
8,active,int64,138,0.0,2,2
9,approval_status,object,138,0.0,2,waiting


Kosong semua: ['bbm']
Konstan     : ['bbm', 'created_at', 'created_by']
Tersimpan: porttoport_cleansing_check.xlsx | 138 baris x 12 kolom
